### Load data and set up/clean

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("williamscott701/memotion-dataset-7k")

In [2]:
###############################################################################
# 1) Install and Imports
###############################################################################
!pip install torch torchvision transformers scikit-learn

import os
import copy
import numpy as np
import pandas as pd
from PIL import Image, ImageFile

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split

from transformers import CLIPProcessor, CLIPModel
###############################################################################
# We allow truncated images just in case
ImageFile.LOAD_TRUNCATED_IMAGES = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

###############################################################################
# 2) Load & Clean Data
###############################################################################
CSV_PATH = "/kaggle/input/memotion-dataset-7k/memotion_dataset_7k/labels.csv"
IMAGES_DIR = "/kaggle/input/memotion-dataset-7k/memotion_dataset_7k/images"

df = pd.read_csv(CSV_PATH)
df = df[["image_name", "text_corrected", "overall_sentiment"]].dropna(subset=["text_corrected", "overall_sentiment"])
df["text_corrected"] = df["text_corrected"].astype(str)
df = df[df["text_corrected"].str.strip() != ""]

# Verify images, skip any fully corrupt
bad_images = 0
valid_indices = []
for i, row in df.iterrows():
    img_path = os.path.join(IMAGES_DIR, row["image_name"])
    try:
        with Image.open(img_path) as im:
            im.verify()
        valid_indices.append(i)
    except:
        bad_images += 1
df = df.loc[valid_indices].reset_index(drop=True)
print(f"Skipped {bad_images} corrupt images. Valid images: {len(df)}")

Using device: cuda
Skipped 1 corrupt images. Valid images: 6986


### Merge to 3 classes and do stratified split

In [3]:
###############################################################################
# 3) Merge 5 Original Classes into 3 (positive, negative, neutral)
###############################################################################
def merge_sentiment(label):
    if label in ["positive", "very_positive"]:
        return "positive"
    elif label in ["negative", "very_negative"]:
        return "negative"
    else:
        return "neutral"

df["merged_sentiment"] = df["overall_sentiment"].apply(merge_sentiment)
label2id = {"positive": 0, "negative": 1, "neutral": 2}
df["label"] = df["merged_sentiment"].map(label2id)
print("Merged distribution:")
print(df["merged_sentiment"].value_counts(normalize=True))

###############################################################################
# 4) Stratified Split
###############################################################################
train_df, test_df = train_test_split(
    df, test_size=0.2, stratify=df["label"], random_state=42
)
val_df, test_df = train_test_split(
    test_df, test_size=0.5, stratify=test_df["label"], random_state=42
)

print("Train/Val/Test sizes:", len(train_df), len(val_df), len(test_df))

Merged distribution:
merged_sentiment
positive    0.594761
neutral     0.314916
negative    0.090324
Name: proportion, dtype: float64
Train/Val/Test sizes: 5588 699 699


### Set up CLIP model

In [4]:
###############################################################################
# 5) CLIP Processor
###############################################################################
# We'll use the openai/clip-vit-base-patch32 variant 
clip_model_name = "openai/clip-vit-base-patch32"
processor = CLIPProcessor.from_pretrained(clip_model_name)
# This processor will handle image transforms + text tokenization automatically
# We'll apply it inside our custom dataset.

###############################################################################
# 6) Custom Dataset
###############################################################################
class CLIPMemotionDataset(Dataset):
    """
    Each item returns:
      - pixel_values: The transformed image tensor
      - input_ids, attention_mask: tokenized text
      - label: the sentiment class
    """
    def __init__(self, dataframe, images_dir, processor, max_length=77):
        self.df = dataframe.reset_index(drop=True)
        self.images_dir = images_dir
        self.processor = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.images_dir, row["image_name"])
        image = Image.open(img_path).convert("RGB")
        text = row["text_corrected"]
        label = torch.tensor(row["label"], dtype=torch.long)

        # The CLIP processor can handle both images & text in a single call,
        # but we'll call it separately for clarity. We'll do them in one go:
        encoded = self.processor(
            text=[text],
            images=[image],
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        # encoded is a dict with keys: pixel_values, input_ids, attention_mask
        # shape: (batch=1, channels/HW or tokens)

        # We'll squeeze out batch=1 dimension so we can return plain tensors
        pixel_values = encoded["pixel_values"].squeeze(0)       # [3, 224, 224]
        input_ids = encoded["input_ids"].squeeze(0)             # [max_length]
        attention_mask = encoded["attention_mask"].squeeze(0)   # [max_length]

        return {
            "pixel_values": pixel_values,
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "label": label
        }

train_dataset = CLIPMemotionDataset(train_df, IMAGES_DIR, processor)
val_dataset   = CLIPMemotionDataset(val_df,   IMAGES_DIR, processor)
test_dataset  = CLIPMemotionDataset(test_df,  IMAGES_DIR, processor)

from torch.utils.data import DataLoader

BATCH_SIZE = 8
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

###############################################################################
# 7) CLIP Classification Model
###############################################################################
class CLIPClassifier(nn.Module):
    """
    Wraps a CLIPModel and adds a small classifier for 3-class sentiment.
    We'll:
      - get image_embeds from model outputs
      - get text_embeds from model outputs
      - combine them, then pass through a small feedforward
    """
    def __init__(self, model_name, num_labels=3, freeze_clip=False):
        super().__init__()
        self.clip_model = CLIPModel.from_pretrained(model_name)
        self.num_labels = num_labels

        # Optionally freeze entire CLIP to reduce memory usage & avoid large updates
        if freeze_clip:
            for param in self.clip_model.parameters():
                param.requires_grad = False

        embed_dim = self.clip_model.config.projection_dim * 2  # e.g., 512 + 512 = 1024
        # Add a small classifier head
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_labels)
        )

    def forward(self, pixel_values, input_ids, attention_mask):
        # The CLIP forward pass:
        # returns image_embeds, text_embeds, etc.
        outputs = self.clip_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values
        )
        # outputs.image_embeds: [batch_size, projection_dim] default=512
        # outputs.text_embeds:  [batch_size, projection_dim]

        # By default, CLIPModel output embeddings are already normalized (unit sphere)
        image_embeds = outputs.image_embeds
        text_embeds = outputs.text_embeds

        # Concatenate them for classification
        fused = torch.cat([image_embeds, text_embeds], dim=1)  # shape: [B, 1024]
        logits = self.classifier(fused)
        return logits

num_labels = 3
model = CLIPClassifier(
    model_name=clip_model_name, 
    num_labels=num_labels, 
    freeze_clip=False  
).to(DEVICE)

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

### Set up optimizer and learning rate scheduler

In [5]:
###############################################################################
# 8) Optimizer / Scheduler
###############################################################################
# Fine-tuning CLIP can be costly. We do a small LR.
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
# Tried both simple scheduler and ReduceLROnPlateau
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=1, verbose=True)

criterion = nn.CrossEntropyLoss()

### Set up training loop and evaluation metrics

In [6]:
###############################################################################
# 9) Training / Evaluation Functions
###############################################################################
def epoch_step(model, dataloader, is_train=False):
    if is_train:
        model.train()
    else:
        model.eval()

    total_loss = 0
    all_preds = []
    all_labels = []

    for batch in dataloader:
        pixel_values = batch["pixel_values"].to(DEVICE)
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        if is_train:
            optimizer.zero_grad()

        logits = model(
            pixel_values=pixel_values,
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        loss = criterion(logits, labels)
        total_loss += loss.item() * labels.size(0)

        preds = logits.argmax(dim=1).detach().cpu().numpy()
        all_preds.append(preds)
        all_labels.append(labels.detach().cpu().numpy())

        if is_train:
            loss.backward()
            optimizer.step()

    avg_loss = total_loss / len(dataloader.dataset)
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    acc = accuracy_score(all_labels, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="macro")
    return avg_loss, acc, prec, rec, f1




In [7]:
###############################################################################
# 10) Training Loop (with early stopping)
###############################################################################
EPOCHS = 5
patience = 2
best_val_loss = float("inf")
no_improve = 0
best_state = None

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc, train_prec, train_rec, train_f1 = epoch_step(model, train_loader, is_train=True)
    val_loss, val_acc, val_prec, val_rec, val_f1 = epoch_step(model, val_loader, is_train=False)

    # Step scheduler on val_loss
    scheduler.step(val_loss)

    print(f"\nEpoch {epoch}/{EPOCHS}")
    print(f"  Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | P: {train_prec:.4f} | R: {train_rec:.4f} | F1: {train_f1:.4f}")
    print(f"  Val   Loss: {val_loss:.4f}   | Acc: {val_acc:.4f}   | P: {val_prec:.4f} | R: {val_rec:.4f} | F1: {val_f1:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        no_improve = 0
        best_state = copy.deepcopy(model.state_dict())
    else:
        no_improve += 1
        if no_improve >= patience:
            print("Early stopping triggered.")
            break

if best_state:
    model.load_state_dict(best_state)

###############################################################################
# Final Test
###############################################################################
test_loss, test_acc, test_prec, test_rec, test_f1 = epoch_step(model, test_loader, is_train=False)
print("\n--- CLIP Test Results ---")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Acc:  {test_acc:.4f}")
print(f"Macro Precision: {test_prec:.4f}")
print(f"Macro Recall:    {test_rec:.4f}")
print(f"Macro F1:        {test_f1:.4f}")

/usr/local/lib/python3.10/dist-packages/PIL/Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))



Epoch 1/5
  Train Loss: 0.9406 | Acc: 0.5906 | P: 0.3207 | R: 0.3346 | F1: 0.2612
  Val   Loss: 0.8989   | Acc: 0.5951   | P: 0.1984 | R: 0.3333 | F1: 0.2487


/usr/local/lib/python3.10/dist-packages/PIL/Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))



Epoch 2/5
  Train Loss: 0.8948 | Acc: 0.5947 | P: 0.1982 | R: 0.3333 | F1: 0.2486
  Val   Loss: 0.8903   | Acc: 0.5951   | P: 0.1984 | R: 0.3333 | F1: 0.2487


/usr/local/lib/python3.10/dist-packages/PIL/Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))



Epoch 3/5
  Train Loss: 0.8914 | Acc: 0.5947 | P: 0.1982 | R: 0.3333 | F1: 0.2486
  Val   Loss: 0.8958   | Acc: 0.5951   | P: 0.1984 | R: 0.3333 | F1: 0.2487


/usr/local/lib/python3.10/dist-packages/PIL/Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))



Epoch 4/5
  Train Loss: 0.8866 | Acc: 0.5947 | P: 0.1982 | R: 0.3333 | F1: 0.2486
  Val   Loss: 0.8997   | Acc: 0.5951   | P: 0.1984 | R: 0.3333 | F1: 0.2487
Early stopping triggered.

--- CLIP Test Results ---
Test Loss: 0.8904
Test Acc:  0.5951
Macro Precision: 0.1984
Macro Recall:    0.3333
Macro F1:        0.2487


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
